Import library

In [33]:
import os
import sys
from tqdm import tqdm
import glob

# Ensure the config module is imported from the project codes directory
PROJECT_CODES_DIR = "/home2/ducvu/cogno-speak/codes"
if PROJECT_CODES_DIR not in sys.path:
    sys.path.insert(0, PROJECT_CODES_DIR)

import importlib
import config_classifiers
config_classifiers = importlib.reload(config_classifiers)

import numpy as np
import pandas as pd
from collections import defaultdict

from sklearn.preprocessing import binarize,scale, robust_scale
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.svm import SVR, SVC, LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score, roc_auc_score, r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


In [34]:
def read_file(file_path):
    f = open(file_path, 'r')
    return f.read()

In [35]:
def calcualte_metrics(actual_labels, pred_vals, avg=None):
    if avg == None:
        f1_val, pres_val, rec_val, conf_val = score(actual_labels, pred_vals, average=None)
    else:
        f1_val, pres_val, rec_val, conf_val = score(actual_labels, pred_vals, average=avg)
    return f1_val, pres_val, rec_val, conf_val

In [36]:
def majority_voting_labels(df, a_class_type, verbose):
    data = {
        "r_IDs": df[["r_IDs", "pred_label"]].groupby("r_IDs").mean().index.values,
        "grouped_pred_label": df[["r_IDs", "pred_label"]].groupby("r_IDs").mean().pred_label.values
    }
    df_final_test_grouped = df.merge(pd.DataFrame(data), on="r_IDs", how="inner")
    print(f"df_final_test_grouped: {df_final_test_grouped}")

    df_final_test_grouped = df_final_test_grouped.drop_duplicates(subset=["r_IDs"], keep="first")
    print(f"df_final_test_grouped_without_duplicates: {df_final_test_grouped}")

    if a_class_type == "3-way":
        threshold_val = 0.3333333
        final_pred_label_list = []
        for x in df_final_test_grouped.grouped_pred_label:
            if x < threshold_val:
                final_pred_label_list.append(0)
            elif x >= threshold_val and x < (2 * threshold_val):
                final_pred_label_list.append(1)
            else:
                final_pred_label_list.append(2)
        df_final_test_grouped["pred_label"] = final_pred_label_list
    else:
        threshold_val = 0.5
        final_pred_label_list = []
        for x in df_final_test_grouped.grouped_pred_label:
            if x < threshold_val:
                final_pred_label_list.append(0)
            else:
                final_pred_label_list.append(1)
    df_final_test_grouped.insert(len(df_final_test_grouped.columns), "final_pred_label", final_pred_label_list)


    actual_labels = df_final_test_grouped.labels

    # Calculate the initial results
    col_2_cons = "final_pred_label"
    pred_vals = df_final_test_grouped["col_2_cons"]
    f1_val, pres_val, rec_val, conf_val = calcualte_metrics(actual_labels, pred_vals, avg="macro")
    
    if verbose == 1:
        print(f"Macro F1-score: {round(f1_val, 2)}")
        print(f"Macro Precision: {round([pres_val, 2])}")
        print(f"Macro Recall: {reound(rec_val, 2)}")
        print(f"Conf matrix: {conf_val}")
    
    if a_class_type == "3-way":
        precision, recall, fscore, support = score(actual_labels, pred_vals)
        if verbose == 1:
            print("Metric \t HC \t MCI \t Dementia")
            print(f"Precision \t {round(precision[0], 2)} \t {round(precision[1], 2)}, \t {round(precision[2], 2)}")
            print(f"Precision \t {round(recall[0], 2)} \t {round(recall[1], 2)}, \t {round(recall[2], 2)}")
            print(f"Precision \t {round(fscore[0], 2)} \t {round(fscore[1], 2)}, \t {round(fscore[2], 2)}")

    return f1_val, pres_val, rec_val, conf_val

In [58]:
def analyze_dataset(df_metadata_final):
    stat_result = defaultdict()

    diagnosis_group = df_metadata_final.diagnosis.unique()

    # Diagnosis distribution
    diagnosis_count = df_metadata_final.diagnosis.value_counts()
    print(f"All data: {len(df_metadata_final)} and diagnosis count: {diagnosis_count}")

    # Age distribution
    age_stat = defaultdict()
    for diagnosis in diagnosis_group:
        group_data = df_metadata_final[df_metadata_final.diagnosis == diagnosis].age
        mean_age = np.mean(group_data)
        std_age = np.std(group_data)

        age_stat[diagnosis] = {
            "Mean age": mean_age,
            "Std age": std_age,
        }

    age_stat["Overall"] = {
        "Mean age": np.mean(df_metadata_final.age),
        "Std age": np.std(df_metadata_final.age),
    }

    bin_range = (0, 50, 60, 70, 80, 100)
    bin_labels = ["0 - 50", "50 - 60", "60 - 70", "70 - 80", "80 - 100"]

    age_histograms = defaultdict()
    for diagnosis in diagnosis_group:
        group_data = df_metadata_final[df_metadata_final.diagnosis == diagnosis].age
        hist, _ = np.histogram(group_data, bins=bin_range)

        age_histograms[diagnosis] = dict(zip(bin_labels, hist))

    stat_result["age stat"] = age_stat
    stat_result["age histograms"] = age_histograms
    # Ethnicity Distribution in disease
    ethnic_diagnosis = defaultdict()
    ethnic_group = df_metadata_final.ethnicity.unique()
    for ethnic in ethnic_group:
        diagnosis_count = df_metadata_final[df_metadata_final.ethnicity == ethnic].diagnosis.value_counts()
        ethnic_diagnosis[ethnic] = diagnosis_count

    stat_result["Ethnic diagnosis"] = ethnic_diagnosis

    # Gender distribution
    gender_diagnosis = defaultdict()
    for diagnosis in diagnosis_group:
        gender_count = df_metadata_final[df_metadata_final.diagnosis == diagnosis].age.value_counts()
        gender_diagnosis[diagnosis] = gender_count

    stat_result["gender diagnosis"] = gender_diagnosis


    return stat_result

In [ ]:
def acoustic_extraction():

In [ ]:
# Hyperparameter
CV_SCORER = config_classifiers.CV_SCORER
N_FOLDS = config_classifiers.N_FOLDS

# Directory
DATA_PATH = config_classifiers.DATA_PATH
FEATS_PATH = config_classifiers.FEATS_PATH
RESULTS_PATH = config_classifiers.RESULTS_PATH

# Tasks
LIST_TASKS = config_classifiers.LIST_TASKS
LIST_ACOUSTIC = config_classifiers.LIST_ACOUSTIC_TYPE

In [60]:
df_metadata_final = pd.read_csv(f"{DATA_PATH}/metadata.csv")
print(df_metadata_final)

    participant_id  age gender diagnosis      ethnicity  labels FOLD_0 FOLD_1  \
0  participant_001   62      F        HC  White-British       0  TRAIN   TEST   
1  participant_002   58      M        HC  White-British       0  TRAIN  TRAIN   
2  participant_003   69      F        HC          Asian       0   TEST  TRAIN   
3  participant_004   64      M        HC          Other       0  TRAIN  TRAIN   
4  participant_005   71      F        HC  White-British       0  TRAIN  TRAIN   
5  participant_006   78      M  Dementia  White-British       1  TRAIN   TEST   
6  participant_007   82      F  Dementia  White-British       1  TRAIN  TRAIN   
7  participant_008   75      M  Dementia          Asian       1   TEST  TRAIN   
8  participant_009   80      F  Dementia          Mixed       1  TRAIN  TRAIN   
9  participant_010   77      M  Dementia  White-British       1  TRAIN  TRAIN   

  FOLD_2 FOLD_3 FOLD_4  
0  TRAIN  TRAIN  TRAIN  
1   TEST  TRAIN  TRAIN  
2  TRAIN  TRAIN  TRAIN  
3  TRAIN

In [62]:
stat_result = analyze_dataset(df_metadata_final)

All data: 10 and diagnosis count: diagnosis
HC          5
Dementia    5
Name: count, dtype: int64
